# 🏆 Laboratorio de ML Supervisado: quince modelos compiten por su negocio

**Diplomado en Inteligencia Artificial · UNAC · Clase 04 · Plantilla del aula**

### 🎯 Qué vamos a hacer
- Responder una pregunta de negocio real: **¿este producto se agota esta semana en el punto de venta?**
- Poner a **competir ~15 modelos de Machine Learning** con una sola orden, y quedarnos con el campeón.
- Ver **dónde se equivoca** el campeón (la matriz de confusión) y **qué variables** mueven su decisión.
- **Guardar el modelo entrenado** en un archivo y usarlo con un caso nuevo — sin volver a entrenar.

### 🧭 Cómo se usa
- Usted **no escribe código**. Corre las celdas con el botón de ▶️, o todas juntas con
  **Entorno de ejecución → Ejecutar todas**.
- Lo único que se cambia está en el **PANEL DE CONTROL** (más abajo, marcado con 🎛️).
- La primera vez hay **un reinicio obligatorio** después de la instalación. Está señalizado. No se lo salte.

---
**Ficha del experimento** — *un experimento sin ficha no se puede repetir ni defender ante un jefe:*

| | |
|---|---|
| **Pregunta de negocio** | ¿Se agota el producto esta semana? (sí / no) |
| **Qué entra** | 8 columnas por producto-semana: ventas, precio, promoción, inventario… |
| **Qué sale** | Un modelo entrenado (`.pkl`) + su reporte de acierto y errores |
| **Semilla** | 42 — para que a todo el salón le den los mismos números |

## ⚙️ Paso 0 · Instalar el laboratorio (una sola vez por sesión)

Usamos **PyCaret**, el laboratorio de experimentos de Machine Learning. En Colab se instala
con el paquete `pycaret-core` (el fork comunitario vigente — **no** use `pip install pycaret`,
que ya no es compatible con el Python de Colab).

La celda siguiente tarda **2 a 4 minutos**. Déjela terminar.

In [ ]:
# Instalar PyCaret (fork comunitario pycaret-core). Tarda 2-4 minutos.
!pip install -q pycaret-core
print("Instalación terminada. AHORA VIENE EL PASO MÁS IMPORTANTE: REINICIAR. ⬇️ Lea la celda siguiente.")

### 🔁 REINICIE AQUÍ — es normal y es obligatorio

La instalación cambió piezas internas de Colab, y Colab necesita reiniciarse para usarlas.

1. Menú **Entorno de ejecución → Reiniciar sesión** (o *Runtime → Restart session*).
2. Confirme el aviso.
3. Continúe **desde la celda siguiente** (▶️ en la de verificación).
4. **NO vuelva a correr la celda de instalación** — ya quedó instalada.

> Si aparece un aviso de "sesión falló" o "se reinició el entorno": tranquilo, eso es
> exactamente lo que pedimos que pasara.

In [ ]:
# ✅ Verificación: ¿quedó listo el laboratorio?
try:
    import pycaret
    print(f"Todo listo. PyCaret {pycaret.__version__} funcionando. Siga a la sección 1️⃣.")
except ImportError:
    print("Aún no está listo. Dos causas posibles:")
    print("  1. No corrió la celda de instalación de arriba → córrala y luego reinicie.")
    print("  2. Instaló pero NO reinició → Entorno de ejecución → Reiniciar sesión, y vuelva aquí.")

## 1️⃣ ¿Qué problema vamos a resolver?

Un supermercado pierde plata de dos formas opuestas: si el producto **se agota**, pierde la
venta y el cliente; si pide **de más**, pierde margen y bodega. La pregunta que vale plata:
**¿cuáles productos se van a agotar esta semana?**

Tenemos la historia de 1.200 combinaciones producto-semana, **ya respondidas**: sabemos si se
agotaron o no. Eso es Machine Learning **supervisado**: aprender de ejemplos con respuesta.

Los datos de esta plantilla son **sintéticos** (inventados con un patrón realista) para que el
laboratorio corra sin depender de nada. Al final le mostramos cómo poner los **suyos**.

**Qué mirar en el resultado:** una tabla con 8 columnas de negocio y la columna
`se_agota` — la respuesta que el modelo va a aprender a predecir.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1200

datos = pd.DataFrame({
    "ventas_semana_pasada":  rng.poisson(60, n) + rng.integers(0, 40, n),
    "ventas_hace_2_semanas": rng.poisson(58, n) + rng.integers(0, 40, n),
    "precio":                np.round(rng.uniform(2.0, 30.0, n), 1),
    "descuento_pct":         rng.choice([0, 0, 0, 10, 20, 30], n),
    "es_quincena":           rng.choice([0, 1], n, p=[0.7, 0.3]),
    "hay_festivo":           rng.choice([0, 1], n, p=[0.85, 0.15]),
    "inventario_inicial":    rng.integers(20, 220, n),
    "dias_desde_reabasto":   rng.integers(1, 15, n),
})

# La "realidad" que el modelo debe descubrir: demanda alta + poco inventario = se agota.
empuje = (
    0.030 * datos["ventas_semana_pasada"]
    + 0.012 * datos["ventas_hace_2_semanas"]
    + 0.045 * datos["descuento_pct"]
    + 0.90  * datos["es_quincena"]
    + 0.70  * datos["hay_festivo"]
    - 0.022 * datos["inventario_inicial"]
    + 0.10  * datos["dias_desde_reabasto"]
    - 2.30
)
probabilidad = 1 / (1 + np.exp(-empuje + rng.normal(0, 1.1, n)))   # ruido: la vida real no es perfecta
datos["se_agota"] = np.where(probabilidad > 0.5, "Si", "No")

# Para el reto final: la misma realidad, en tres niveles de riesgo
datos["nivel_riesgo"] = pd.cut(probabilidad, [0, 0.35, 0.65, 1.0],
                               labels=["bajo", "medio", "alto"]).astype(str)

print(f"{len(datos)} filas · asi se reparten las respuestas:")
print(datos["se_agota"].value_counts().to_string())
datos.head(8)

## 2️⃣ 🎛️ PANEL DE CONTROL

> **AQUÍ Y SOLO AQUÍ SE CAMBIAN COSAS.** Todo lo demás corre solo.

- **COLUMNA_OBJETIVO**: la pregunta que el modelo aprende (`se_agota`, o `nivel_riesgo` para el reto final).
- **PORCENTAJE_ENTRENAMIENTO**: con cuántos datos estudia (el resto queda para el examen).
- **SEMILLA**: fíjela y todo el salón obtiene los mismos números. Cámbiela y verá bailar los decimales.

In [ ]:
COLUMNA_OBJETIVO = "se_agota"  #@param ["se_agota", "nivel_riesgo"]
PORCENTAJE_ENTRENAMIENTO = 0.8  #@param {type:"slider", min:0.6, max:0.9, step:0.05}
SEMILLA = 42  #@param {type:"integer"}

# la columna de la "otra" pregunta se aparta para no hacer trampa
columnas_a_ignorar = [c for c in ["se_agota", "nivel_riesgo"] if c != COLUMNA_OBJETIVO]
print(f"Pregunta a aprender: {COLUMNA_OBJETIVO}  ·  estudia con {PORCENTAJE_ENTRENAMIENTO:.0%} y se examina con {1-PORCENTAJE_ENTRENAMIENTO:.0%}")

## 3️⃣ Miremos los datos antes de modelar (siempre)

Regla de la casa: **nunca entrene con datos que no ha mirado.** Dos vistazos: los números
resumidos, y los valores atípicos (errores de digitación, casos rarísimos) que pueden
desviar el aprendizaje. Aquí los tratamos con una función de una línea: los valores fuera de
rango se reemplazan por la mediana **de su propio grupo**.

**Qué mirar en el resultado:** en el par de cajas (antes / después), las colas largas de
puntos desaparecen sin mover el centro de los datos.

In [ ]:
import matplotlib.pyplot as plt

def limpiar_atipicos(df, columna_grupo):
    """Reemplaza atipicos (regla IQR) por la mediana de su grupo, columna por columna."""
    limpio = df.copy()
    numericas = limpio.select_dtypes("number").columns
    for grupo, bloque in limpio.groupby(columna_grupo):
        for col in numericas:
            q1, q3 = bloque[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            fuera = (bloque[col] < q1 - 1.5 * iqr) | (bloque[col] > q3 + 1.5 * iqr)
            limpio.loc[bloque.index[fuera], col] = bloque[col].median()
    return limpio

datos_limpios = limpiar_atipicos(datos, COLUMNA_OBJETIVO)

fig, ejes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
ejes[0].boxplot(datos["ventas_semana_pasada"]);        ejes[0].set_title("Antes de limpiar")
ejes[1].boxplot(datos_limpios["ventas_semana_pasada"]); ejes[1].set_title("Después de limpiar")
for e in ejes: e.set_ylabel("ventas de la semana pasada")
plt.suptitle("Valores atípicos: antes vs. después")
plt.tight_layout(); plt.show()

datos_limpios.describe().round(1)

## 4️⃣ Preparar la mesa de experimentos (`setup`)

Tres decisiones, con su analogía:
- **80 / 20**: el modelo *estudia* con 8 de cada 10 casos y se *examina* con los 2 que nunca vio.
- **fold = 5**: el examen se repite 5 veces con grupos distintos, para que el acierto no sea suerte.
- **normalizar**: poner todas las columnas en la misma escala, para que ninguna mande solo por ser grande.

**Qué mirar en el resultado:** la tabla-resumen de PyCaret confirmando cuántas filas van a
entrenamiento y cuántas al examen.

In [ ]:
from pycaret.classification import *

experimento = setup(
    data=datos_limpios,
    target=COLUMNA_OBJETIVO,           # la pregunta
    ignore_features=columnas_a_ignorar,  # la otra pregunta se aparta (no vale hacer trampa)
    train_size=PORCENTAJE_ENTRENAMIENTO,
    session_id=SEMILLA,                # la semilla: reproducible
    normalize=True,                    # misma escala para todas las columnas
    fold=5,                            # 5 exámenes cruzados
    use_gpu=False,
    verbose=False,
)
print("Mesa lista. Filas para estudiar:", experimento.X_train.shape[0],
      "· filas para el examen final:", experimento.X_test.shape[0])

## 5️⃣ ⭐ La competencia: ~15 modelos, un podio

Esta es LA celda del laboratorio. Con **una sola orden**, PyCaret entrena todos los modelos
del catálogo — regresión logística, árboles, bosques, gradient boosting y compañía — les hace
el mismo examen 5 veces a cada uno, y arma el podio.

Usted no elige el algoritmo a mano: **los manda a competir.** Tarda 1–3 minutos.

**Qué mirar en el resultado:** la primera fila es el campeón. La columna **Accuracy** es el
acierto (0.88 = acierta 88 de cada 100). Si un modelo marcara **1.00, desconfíe**: nadie se
saca 100 en un examen de verdad — casi siempre significa que algo se filtró.

In [ ]:
campeon_bruto = compare_models(sort="Accuracy")  # ⭐ la orden que reemplaza semanas de trabajo


## 6️⃣ Apretarle las tuercas al campeón (`tune_model`)

Cada modelo tiene perillas internas (hiperparámetros). Esta celda prueba 10 combinaciones y
se queda con la mejor — y si ninguna supera al original, **devuelve el original** (no
empeoramos por probar).

**Qué mirar en el resultado:** compare el Accuracy promedio de esta tabla con el del podio.
Subir 1 punto aquí es normal; subir 10 es sospechoso.

In [ ]:
campeon = tune_model(campeon_bruto, n_iter=10, optimize="Accuracy", choose_better=True)


## 7️⃣ ¿Dónde se equivoca? — la matriz de confusión

La gráfica más importante para un gerente. Cuatro casillas, dos errores con **costos
distintos**:

| | El modelo dijo **Sí** | El modelo dijo **No** |
|---|---|---|
| **De verdad se agotó** | ✅ Alarma correcta | 🔴 **El estante quedó vacío** (venta y cliente perdidos) |
| **No se agotó** | 🟡 Falsa alarma (se pidió de más) | ✅ Tranquilidad correcta |

En este negocio, el error 🔴 suele costar más que el 🟡. Ese juicio es **suyo**, no del modelo
— y en la próxima clase aprenderemos a mover el modelo hacia el error que menos duela.

**Qué mirar en el resultado:** ¿cuántos casos caen en la casilla roja?

In [ ]:
plot_model(campeon, plot="confusion_matrix")

### ¿Y qué variables mueven la decisión?

**Qué mirar en el resultado:** las 2–3 barras más largas. ¿Coinciden con la intuición del
negocio (inventario bajo + ventas altas)? Si una variable absurda domina, sospeche de los datos.

In [ ]:
try:
    plot_model(campeon, plot="feature")
except Exception:
    print("Este modelo no expone importancia de variables (les pasa a algunos, como KNN).")
    print("Corra en su lugar:  plot_model(campeon, plot='class_report')")

## 8️⃣ El examen final: los datos que nunca vio

Hasta ahora todo se midió con validación cruzada **dentro** del estudio. Esta celda evalúa al
campeón sobre el **20 % apartado desde el principio** — preguntas de examen que jamás vio.
Estos son los números que se reportan.

**Qué mirar en el resultado:** que el Accuracy del examen esté cerca del de la competencia.
Si en el estudio sacaba 95 y en el examen 75, el modelo **memorizó** en vez de aprender:
eso se llama sobreajuste.

In [ ]:
resultado_examen = predict_model(campeon)
resultado_examen[[c for c in resultado_examen.columns if c in
                  ("ventas_semana_pasada", "inventario_inicial", COLUMNA_OBJETIVO,
                   "prediction_label", "prediction_score")]].head(10)

## 9️⃣ Guardar el cerebro — y usarlo mañana sin volver a entrenar

El modelo entrenado se guarda en **un archivo** (`.pkl`). Eso lo convierte en un activo de su
empresa: mañana se carga y predice en segundos. Aquí mismo lo probamos con **un caso nuevo
escrito a mano** — cámbiele los números y vuelva a correr la celda.

**Qué mirar en el resultado:** la decisión (`prediction_label`) y la confianza
(`prediction_score`) para su caso inventado.

In [ ]:
modelo_final = finalize_model(campeon)          # se re-entrena con TODOS los datos, ya validado
save_model(modelo_final, "modelo_agotamiento")  # queda el archivo modelo_agotamiento.pkl

caso_nuevo = pd.DataFrame([{
    "ventas_semana_pasada": 95,   # se vendió mucho...
    "ventas_hace_2_semanas": 88,
    "precio": 6.5,
    "descuento_pct": 20,          # ...está en promoción...
    "es_quincena": 1,             # ...es quincena...
    "hay_festivo": 0,
    "inventario_inicial": 35,     # ...y hay poco inventario.
    "dias_desde_reabasto": 9,
}])

cerebro = load_model("modelo_agotamiento")
veredicto = predict_model(cerebro, data=caso_nuevo)
etiqueta = veredicto["prediction_label"].iloc[0]
confianza = veredicto["prediction_score"].iloc[0]
print(f"\nVeredicto para el caso nuevo →  {COLUMNA_OBJETIVO} = {etiqueta}  (confianza {confianza:.0%})")

### 💾 Para llevarse el archivo a su computador (opcional)

Corra esta celda si quiere descargar el modelo entrenado.

In [ ]:
try:
    from google.colab import files
    files.download("modelo_agotamiento.pkl")
except Exception:
    print("Esta descarga solo funciona dentro de Google Colab.")

## 🔬 Para curiosos (opcional): ¿por qué decidió lo que decidió?

SHAP muestra, caso por caso, qué variable empujó la decisión hacia el Sí o hacia el No.
Es la respuesta moderna a "la IA es una caja negra". Instala una librería extra; si falla,
sáltela sin culpa.

In [ ]:
%pip install -q shap
try:
    interpret_model(campeon, plot="summary")
except Exception as e:
    print("SHAP solo funciona con modelos de árbol (bosque, gradient boosting).")
    print(f"Detalle técnico: {e}")

## 🔁 El reto: de dos clases a tres

Los **mismos datos** responden dos preguntas distintas:
- `se_agota` → **¿hay problema?** (sí / no)
- `nivel_riesgo` → **¿de qué tamaño es el problema?** (bajo / medio / alto)

**El reto:** suba al 🎛️ PANEL DE CONTROL, cambie `COLUMNA_OBJETIVO` a `nivel_riesgo`, y corra
todo de nuevo desde la sección 3️⃣. Mismo laboratorio, otra pregunta, otro modelo.
Así funciona esto en la vida real: el laboratorio queda montado y las preguntas van cambiando.

## 🏁 Ahora con SUS datos

Esta plantilla se vuelve suya cambiando **una sola cosa**: la tabla de datos.

1. Prepare un Excel/CSV donde **cada fila es un caso** ya resuelto (un cliente, una factura,
   un producto-semana) y **una columna tiene la respuesta** (se fue / se quedó · pagó / no pagó ·
   se agotó / no). Entre 300 y 5.000 filas funciona muy bien.
2. En Colab, abra el panel de archivos (📁 a la izquierda) y arrastre su archivo, **o** cree una
   celda nueva y escriba: `from google.colab import files; files.upload()`
3. Reemplace la celda de la sección 1️⃣ por: `datos = pd.read_csv("su_archivo.csv")`
4. En el 🎛️ PANEL DE CONTROL escriba el nombre de **su** columna de respuesta. Ejecutar todas.

> ⚠️ **Datos sensibles:** nombres de clientes, historias clínicas, datos personales — se
> anonimizan ANTES de subirlos a Colab. En caso de duda, pregunte en la asesoría.

### 🤖 Pregúntele a su Claude (en la carpeta del aula)

- *"Tengo un Excel con estas columnas: … . Adáptame el laboratorio supervisado del aula para
  predecir la columna … y explícame qué significa cada resultado."*
- *"El laboratorio me dio 0.87 de Accuracy y esta matriz de confusión: … . ¿Qué le digo a mi
  jefe y qué error nos cuesta más plata?"*
- *"¿Este modelo me sirve para la Entrega 2 de mi proyecto de grado? Revísalo contra /entrega-2."*

---
*Diplomado en Inteligencia Artificial · UNAC · Clase 04 · Plantilla del aula.*
*Si algo falla: copie el error completo y péguéselo a su Claude — él se sabe este notebook.*